<a href="https://colab.research.google.com/github/NarenCandy/simple_rag_learning/blob/main/RAG_Module2_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 RAG Module 2 — Build Your First RAG Pipeline

**What you'll build:** A Document Q&A bot — feed it any PDF, ask questions, get grounded answers.

**Stack:**
- 🔗 LangChain — orchestration
- 🤗 HuggingFace Embeddings — FREE, no API key needed
- 🗄️ ChromaDB — vector store
- ⚡ Groq API — FREE LLM (super fast)

---
### 📋 Before You Start
1. Get a **FREE Groq API key** → https://console.groq.com (takes 1 min)
2. That's it! Everything else is free and runs in Colab.

---

## 📦 Step 1 — Install Dependencies
Run this cell first. It takes ~1-2 minutes.

In [ ]:
# Install all required packages
!pip install -q langchain langchain-community langchain-groq
!pip install -q chromadb
!pip install -q pypdf
!pip install -q sentence-transformers
!pip install -q groq

print('✅ All packages installed!')

In [ ]:
!pip install -q langchain langchain-community langchain-groq

## 🔑 Step 2 — Set Your FREE Groq API Key
Get your free key at https://console.groq.com → Sign up → API Keys → Create Key

In [ ]:
import os

# Paste your Groq API key here
os.environ['GROQ_API_KEY'] = 'your_api_key'  # 👈 Replace this!

print('✅ API key set!')

## 📄 Step 3 — Upload or Use a Sample PDF

**Option A:** Upload your own PDF using the Colab file panel (folder icon on left)

**Option B:** We'll download a sample PDF (an ML research paper) to practice with

In [ ]:
# Option B: Download a sample PDF (Attention Is All You Need — the famous Transformer paper)
#!wget -q -O sample.pdf "https://arxiv.org/pdf/1706.03762"

PDF_PATH = "/content/Atomic-Habits-.pdf"  # Change this if you uploaded your own PDF

print(f'✅ Using PDF: {PDF_PATH}')

## 📖 Step 4 — Load the Document

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Load the PDF
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

print(f'✅ Loaded {len(documents)} pages from PDF')
print(f'\n📄 Preview of page 1:')
print(documents[0].page_content[:500])

## ✂️ Step 5 — Chunk the Document

We split the document into smaller overlapping chunks so the LLM can handle them.

In [ ]:
!pip install langchain


In [ ]:
import langchain
print(langchain.__version__)


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # max characters per chunk
    chunk_overlap=50     # overlap to avoid losing context at boundaries
)

chunks = splitter.split_documents(documents)

print(f'✅ Created {len(chunks)} chunks from {len(documents)} pages')
print(f'\n🔍 Example chunk:')
print(chunks[5].page_content)
print(f'\n📌 Metadata: {chunks[5].metadata}')

In [ ]:
!pip install -q langchain-huggingface

## 🔢 Step 6 — Embed Chunks & Store in ChromaDB

We use HuggingFace's `all-MiniLM-L6-v2` — a small, fast, FREE embedding model.

⚠️ This cell may take 2-3 minutes on first run (downloads the embedding model).

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print('⏳ Loading embedding model (first time takes ~2 min)...')

# Free HuggingFace embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}  # use 'cuda' if you enable GPU in Colab
)

print('⏳ Embedding chunks and storing in ChromaDB...')

# Embed all chunks and store in ChromaDB
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print(f'✅ Vector store created with {vectorstore._collection.count()} vectors!')

## 🔍 Step 7 — Test the Retriever

Before connecting the LLM, let's verify the retriever finds relevant chunks.

In [ ]:
# Create retriever — fetches top 4 most relevant chunks
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

# Test it with a sample query
test_query = "What is good habits?"
retrieved_docs = retriever.invoke(test_query)

print(f'🔍 Query: "{test_query}"')
print(f'📦 Retrieved {len(retrieved_docs)} chunks:\n')

for i, doc in enumerate(retrieved_docs):
    print(f'--- Chunk {i+1} (Page {doc.metadata.get("page", "?")}) ---')
    print(doc.page_content[:200])
    print()

## 🤖 Step 8 — Connect Groq LLM & Build RAG Chain

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA



# Initialize Groq LLM (free and fast!)
llm = ChatGroq(
    model="llama-3.1-8b-instant",  # free model on Groq
    temperature=0            # 0 = deterministic, factual answers
)

# Custom prompt — tells LLM to use ONLY the provided context
prompt_template = """
You are a helpful assistant. Answer the question using ONLY the context provided below.
If the answer is not in the context, say "I don't have enough information to answer this."

Context:
{context}

Question: {question}

Answer:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# Build the RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)

print('✅ RAG chain is ready!')

## 💬 Step 9 — Ask Questions!

Try asking questions about your PDF. Change the query below.

In [ ]:
def ask(question):
    """Ask a question and get a grounded answer with sources."""
    print(f'❓ Question: {question}')
    print('-' * 60)

    result = qa_chain.invoke({"query": question})

    print(f'💡 Answer:\n{result["result"]}')
    print('\n📌 Sources used:')
    for doc in result['source_documents']:
        page = doc.metadata.get('page', '?')
        print(f'  • Page {page}: {doc.page_content[:120]}...')
    print('=' * 60)


# 👇 Change these questions to match your PDF!
ask("What strategies does the book suggest for breaking a bad habit?")

In [ ]:
ask("how to get better at football?")

In [ ]:
ask("What datasets were used for evaluation?")

In [ ]:
# 🧪 Try your own question!
my_question = "What are the limitations of this approach?"  # 👈 Change this
ask(my_question)

## 🎯 Bonus — Interactive Q&A Loop

Run this cell for a continuous chat experience. Type 'quit' to exit.

In [ ]:
print('🤖 RAG Q&A Bot Ready! Type your questions below.')
print('Type "quit" to stop.\n')

while True:
    question = input('You: ').strip()
    if question.lower() in ['quit', 'exit', 'q']:
        print('👋 Goodbye!')
        break
    if question:
        ask(question)
        print()

In [ ]:
# Fix widget metadata before saving to GitHub
import json
import IPython

# Clear widget state to fix the metadata error
IPython.display.clear_output()

# Run this to strip broken widget metadata
!jupyter nbconvert --to notebook --inplace \
  --ClearMetadataPreprocessor.enabled=True \
  --ClearMetadataPreprocessor.preserve_nb_metadata_keys='["colab","kernelspec","language_info"]' \
  /content/drive/MyDrive/RAG_Module2_Colab.ipynb